In [9]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

from sklearn.model_selection import train_test_split
from sklearn.metrics import explained_variance_score, mean_absolute_error, mean_squared_error, r2_score

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/shubhammehta21/movie-lens-small-latest-dataset/movies.csv
/kaggle/input/datasets/shubhammehta21/movie-lens-small-latest-dataset/ratings.csv
/kaggle/input/datasets/shubhammehta21/movie-lens-small-latest-dataset/README.txt
/kaggle/input/datasets/shubhammehta21/movie-lens-small-latest-dataset/tags.csv
/kaggle/input/datasets/shubhammehta21/movie-lens-small-latest-dataset/links.csv


In [10]:
df = pd.read_csv('/kaggle/input/datasets/shubhammehta21/movie-lens-small-latest-dataset/ratings.csv')

In [11]:
df.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [12]:
df = df.rename(columns = {'userId': 'user_id', 'movieId': 'movie_id'})

In [14]:
df.head()

,user_id,movie_id,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [31]:
df.shape

(100836, 4)

In [32]:
df.describe()

,user_id,movie_id,rating,timestamp
count,100836.000000,100836.000000,100836.000000,1.008360e+05
mean,326.127564,19435.295718,3.501557,1.205946e+09
std,182.618491,35530.987199,1.042529,2.162610e+08
min,1.000000,1.000000,0.500000,8.281246e+08
25%,177.000000,1199.000000,3.000000,1.019124e+09
50%,325.000000,2991.000000,3.500000,1.186087e+09
75%,477.000000,8122.000000,4.000000,1.435994e+09
max,610.000000,193609.000000,5.000000,1.537799e+09


In [13]:
df['user_id'] = df['user_id'] - 1
df['movie_id'] = df['movie_id'] - 1


In [7]:
df['user_id'].min()

0

In [36]:
import numpy as np

class MatrixFactorizationSGD:
    """
    Matrix Factorization with SGD — built directly on your SGDRegressor style
    Predicts rating = user_factor[u] dot item_factor[i]
    """
    def __init__(self, n_factors=20, learning_rate=0.01, epochs=50, 
                 batch_size=1, reg=None, reg_param=0.01):
        self.n_factors = n_factors
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.reg = reg
        self.reg_param = reg_param
        
        self.P = None  # user latent factors
        self.Q = None  # item latent factors
        self.global_bias = 0.0
        
    def fit(self, user_ids, item_ids, ratings):
        """
        user_ids, item_ids, ratings: numpy arrays (or pandas Series) of same length
        """
        user_ids = np.asarray(user_ids, dtype=int)
        item_ids = np.asarray(item_ids, dtype=int)
        ratings = np.asarray(ratings, dtype=float)
        
        n_users = 610
        n_items = 193609
        
        # Initialize latent factors (small random values)
        self.P = np.random.normal(0, 0.01, (n_users, self.n_factors))
        self.Q = np.random.normal(0, 0.01, (n_items, self.n_factors))
        self.global_bias = np.mean(ratings)
        
        for epoch in range(self.epochs):
            # Shuffle (stochastic part)
            indices = np.random.permutation(len(ratings))
            for i in range(0, len(ratings), self.batch_size):
                batch_index = indices[i:i + self.batch_size]
                u_batch = user_ids[batch_index]
                i_batch = item_ids[batch_index]
                r_batch = ratings[batch_index]
                
                # Forward pass
                pred = np.sum(self.P[u_batch] * self.Q[i_batch], axis=1) + self.global_bias
                error = r_batch - pred
                
                # Gradients (same style as your SGDRegressor!)
                grad_P = -2 * (error[:, np.newaxis] * self.Q[i_batch])
                grad_Q = -2 * (error[:, np.newaxis] * self.P[u_batch])
                
                # Regularization (exactly like your code)
                if self.reg == 'l2':
                    grad_P += 2 * self.reg_param * self.P[u_batch]
                    grad_Q += 2 * self.reg_param * self.Q[i_batch]
                elif self.reg == 'l1':
                    grad_P += self.reg_param * np.sign(self.P[u_batch])
                    grad_Q += self.reg_param * np.sign(self.Q[i_batch])
                
                # SGD update
                self.P[u_batch] -= self.learning_rate * grad_P
                self.Q[i_batch] -= self.learning_rate * grad_Q
        
        return self
    
    def predict(self, user_ids, item_ids):
        user_ids = np.asarray(user_ids, dtype=int)
        item_ids = np.asarray(item_ids, dtype=int)
        return np.sum(self.P[user_ids] * self.Q[item_ids], axis=1) + self.global_bias
    

In [37]:
train_df, test_df = train_test_split(df, test_size=0.3, random_state=32, stratify=df['user_id']) 

# Our model expect arrays 
train_users = train_df['user_id'].values
train_movie = train_df['movie_id'].values
train_rating = train_df['rating'].values

test_users = test_df['user_id'].values
test_movie = test_df['movie_id'].values
test_rating = test_df['rating'].values


In [38]:
model = MatrixFactorizationSGD(n_factors=10, learning_rate=0.01, epochs=50, 
                               reg='l1', reg_param=0.02, batch_size=32)

model.fit(train_users, train_movie, train_rating)

In [39]:
test_pred = model.predict(test_users, test_movie)

In [40]:
rmse = np.sqrt(mean_squared_error(test_rating, test_pred))
mae = mean_absolute_error(test_rating, test_pred)
r2score = r2_score(test_rating, test_pred)
explained_var = explained_variance_score(test_rating, test_pred)
print(rmse)
print(mae)
print(r2score)
print(explained_var)

1.0465884948166158
0.8299068281521335
-3.464805321207187e-05
-2.68306399320295e-08
